In [12]:
from pathlib import Path
import sys

# Find project root: the directory containing src/
CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = CURRENT_DIR
while not (PROJECT_DIR / "src").is_dir():
    if PROJECT_DIR.parent == PROJECT_DIR:
        raise FileNotFoundError(
            "Project root not found. Open this notebook from inside the SwiftDrop project."
        )
    PROJECT_DIR = PROJECT_DIR.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_DIR)
print("Models directory:", MODELS_DIR)


Project root: D:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline
Models directory: D:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models


In [13]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import onnx
import onnxruntime as ort
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

torch.set_num_threads(1)
print("PyTorch:", torch.__version__)
print("ONNX Runtime:", ort.__version__)


PyTorch: 2.14.1+cpu
ONNX Runtime: 1.31.0


In [14]:
class DeliveryTimeMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


In [15]:

preprocessor_path = MODELS_DIR / "neural_network_preprocessor.joblib"
weights_path = MODELS_DIR / "neural_network_weights.pth"
validation_path = MODELS_DIR / "nn_validation_data.npz"

for path in (preprocessor_path, weights_path, validation_path):
    if not path.exists():
        raise FileNotFoundError(f"Required artifact not found: {path}")

preprocessor = joblib.load(preprocessor_path)
validation_data = np.load(validation_path)

X_val = np.asarray(
    validation_data["X_val_processed"], dtype=np.float32
)
y_val = np.asarray(
    validation_data["y_val"], dtype=np.float32
).reshape(-1)

# Load the held-out test set for final evaluation
X_test = np.asarray(
    validation_data["X_test_processed"], dtype=np.float32
)
y_test = np.asarray(
    validation_data["y_test"], dtype=np.float32
).reshape(-1)

for split_name, X_split, y_split in [
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
]:
    if X_split.ndim != 2:
        raise ValueError(
            f"{split_name}: expected 2D features, got {X_split.shape}"
        )
    if len(X_split) != len(y_split):
        raise ValueError(
            f"{split_name}: feature/target mismatch "
            f"{len(X_split)} vs {len(y_split)}"
        )

input_dim = X_val.shape[1]
expected_dim = len(preprocessor.get_feature_names_out())

if input_dim != expected_dim or X_test.shape[1] != expected_dim:
    raise ValueError(
        f"Feature mismatch: validation={X_val.shape[1]}, "
        f"test={X_test.shape[1]}, expected={expected_dim}"
    )

model = DeliveryTimeMLP(input_dim)
state_dict = torch.load(
    weights_path, map_location="cpu", weights_only=True
)
model.load_state_dict(state_dict)
model.eval()

print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("Expected feature count:", expected_dim)
print("Saved model and both splits loaded successfully.")


Validation shape: (7793, 46) (7793,)
Test shape: (7793, 46) (7793,)
Expected feature count: 46
Saved model and both splits loaded successfully.


In [16]:
with torch.inference_mode():
    pytorch_preds = model(torch.from_numpy(X_val)).numpy().reshape(-1)

def calculate_metrics(y_true, y_pred):
    return {
        "MAE (minutes)": mean_absolute_error(y_true, y_pred),
        "RMSE (minutes)": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

pytorch_metrics = calculate_metrics(y_val, pytorch_preds)
print("PyTorch validation metrics")
for name, value in pytorch_metrics.items():
    print(f"{name}: {value:.6f}")


PyTorch validation metrics
MAE (minutes): 3.464553
RMSE (minutes): 4.378737
R2: 0.777161


In [17]:
onnx_path = MODELS_DIR / "neural_network.onnx"
dummy_input = torch.zeros((1, input_dim), dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    str(onnx_path),
    input_names=["features"],
    output_names=["prediction"],
    dynamic_axes={
        "features": {0: "batch_size"},
        "prediction": {0: "batch_size"},
    },
    opset_version=17,
    dynamo=False,
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print("ONNX export and structural validation passed:", onnx_path)


ONNX export and structural validation passed: D:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models\neural_network.onnx


C:\Users\dell\AppData\Local\Temp\ipykernel_15516\955769681.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [18]:
session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1
session = ort.InferenceSession(
    str(onnx_path),
    sess_options=session_options,
    providers=["CPUExecutionProvider"],
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
onnx_preds = session.run([output_name], {input_name: X_val})[0].reshape(-1)

max_abs_diff = float(np.max(np.abs(pytorch_preds - onnx_preds)))
mean_abs_diff = float(np.mean(np.abs(pytorch_preds - onnx_preds)))
parity_passed = bool(np.allclose(pytorch_preds, onnx_preds, rtol=1e-4, atol=1e-5))

print("ONNX input:", session.get_inputs()[0].name, session.get_inputs()[0].shape)
print("ONNX output:", session.get_outputs()[0].name, session.get_outputs()[0].shape)
print("Maximum absolute prediction difference:", max_abs_diff)
print("Mean absolute prediction difference:", mean_abs_diff)
print("Predictions match within tolerance:", parity_passed)
if not parity_passed:
    raise AssertionError("PyTorch and ONNX predictions differ beyond tolerance.")


ONNX input: features ['batch_size', 46]
ONNX output: prediction ['batch_size', 1]
Maximum absolute prediction difference: 7.62939453125e-06
Mean absolute prediction difference: 1.2200863466205192e-06
Predictions match within tolerance: True


In [19]:
onnx_metrics = calculate_metrics(y_val, onnx_preds)
comparison = pd.DataFrame(
    [pytorch_metrics, onnx_metrics],
    index=["PyTorch", "ONNX Runtime"],
)
comparison


,MAE (minutes),RMSE (minutes),R2
PyTorch,3.464553,4.378737,0.777161
ONNX Runtime,3.464553,4.378737,0.777161


In [20]:
N_BENCH_ROWS = min(1000, len(X_val))
WARMUP_RUNS = 10
REPEAT_RUNS = 100
X_bench = np.ascontiguousarray(X_val[:N_BENCH_ROWS], dtype=np.float32)
X_bench_tensor = torch.from_numpy(X_bench)

# Warm-up both runtimes
with torch.inference_mode():
    for _ in range(WARMUP_RUNS):
        _ = model(X_bench_tensor)
for _ in range(WARMUP_RUNS):
    _ = session.run([output_name], {input_name: X_bench})

# Repeated PyTorch timing
pytorch_times_ms = []
with torch.inference_mode():
    for _ in range(REPEAT_RUNS):
        start = time.perf_counter()
        _ = model(X_bench_tensor)
        pytorch_times_ms.append((time.perf_counter() - start) * 1000)

# Repeated ONNX Runtime timing
onnx_times_ms = []
for _ in range(REPEAT_RUNS):
    start = time.perf_counter()
    _ = session.run([output_name], {input_name: X_bench})
    onnx_times_ms.append((time.perf_counter() - start) * 1000)

def latency_summary(times_ms, row_count):
    values = np.asarray(times_ms)
    return {
        "Median batch latency (ms)": float(np.median(values)),
        "Mean batch latency (ms)": float(np.mean(values)),
        "P95 batch latency (ms)": float(np.percentile(values, 95)),
        "Median latency per row (ms)": float(np.median(values) / row_count),
    }

latency_results = pd.DataFrame(
    [
        latency_summary(pytorch_times_ms, N_BENCH_ROWS),
        latency_summary(onnx_times_ms, N_BENCH_ROWS),
    ],
    index=["PyTorch CPU", "ONNX Runtime CPU"],
)
print(f"Rows per inference call: {N_BENCH_ROWS}")
print(f"Warm-up runs per runtime: {WARMUP_RUNS}")
print(f"Timed repetitions per runtime: {REPEAT_RUNS}")
latency_results


Rows per inference call: 1000
Warm-up runs per runtime: 10
Timed repetitions per runtime: 100


,Median batch latency (ms),Mean batch latency (ms),P95 batch latency (ms),Median latency per row (ms)
PyTorch CPU,0.6123,0.628812,0.85313,0.000612
ONNX Runtime CPU,0.5665,0.577002,0.65473,0.000566


In [21]:
LATENCY_TARGET_MS = 50.0
onnx_p95_batch_ms = float(latency_results.loc["ONNX Runtime CPU", "P95 batch latency (ms)"])
latency_passed = onnx_p95_batch_ms <= LATENCY_TARGET_MS
print(f"ONNX P95 latency for {N_BENCH_ROWS} rows: {onnx_p95_batch_ms:.3f} ms")
print(f"Target: <= {LATENCY_TARGET_MS:.1f} ms per batch")
print("50 ms batch target passed:", latency_passed)
if not latency_passed:
    print("Note: benchmark on the intended deployment hardware and confirm whether the 50 ms requirement is per request or per batch.")


ONNX P95 latency for 1000 rows: 0.655 ms
Target: <= 50.0 ms per batch
50 ms batch target passed: True


In [22]:
report = {
    "validation_rows": int(len(y_val)),
    "validation_features": int(X_val.shape[1]),
    "pytorch_metrics": {k: float(v) for k, v in pytorch_metrics.items()},
    "onnx_metrics": {k: float(v) for k, v in onnx_metrics.items()},
    "max_abs_prediction_difference": max_abs_diff,
    "mean_abs_prediction_difference": mean_abs_diff,
    "onnx_parity_passed": parity_passed,
    "benchmark_rows": int(N_BENCH_ROWS),
    "warmup_runs": int(WARMUP_RUNS),
    "repeat_runs": int(REPEAT_RUNS),
    "latency_target_ms_per_batch": float(LATENCY_TARGET_MS),
    "onnx_p95_batch_latency_ms": onnx_p95_batch_ms,
    "latency_target_passed": bool(latency_passed),
}
report_path = MODELS_DIR / "onnx_evaluation_report.json"
import json
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Saved report:", report_path)
print("ONNX artifact:", onnx_path)
print("Evaluation complete.")


Saved report: D:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models\onnx_evaluation_report.json
ONNX artifact: D:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models\neural_network.onnx
Evaluation complete.
